# V12B time-ready runner for GUARD10 and FORWARD

This version consumes the validated output of
`V12C_Rebuild_Training_Timestamps.ipynb` instead of trying to infer training
times from feature fingerprints.

Attach three Kaggle inputs: ModelReady_GAPAWARE_V4, the saved V12C timestamp
dataset, and the prior corrected ALL21 results/checkpoints. Run every cell from
the beginning. The default switches run GUARD10 and FORWARD only.


# MBEC V12B — all-configuration and robustness evaluation

V12B starts from the validated V11B implementation and **refits models** from
the existing `ModelReady_GAPAWARE_V4` feature dataset. It does not recompute EEG
features. Five independent stages are provided:

1. `all21`: online-safe evaluation of 3 classifiers × 7 representations;
2. `guard10`: 10-min temporal guard around held-out interictal epochs;
3. `forward`: chronological training on earlier seizures only;
4. `nogamma`: removal of every gamma-band feature from four focal configurations;
5. `multiseed`: focal configurations for seeds 43–46, combined with seed 42
   from `all21` during aggregation.

Every stage writes subject-level checkpoints. A stopped Kaggle session can be
resumed by attaching the saved V12B output dataset, setting `OUTPUT_ROOT` to the
restored folder if necessary, and rerunning with `RESET_OUTPUT=False`.

## Interpretation boundary

The forward stage is pseudo-prospective with respect to both seizure order and
clock time: training, test, and stream rows at or after the target onset are
removed. When a remaining eligible stream epoch belongs to an earlier seizure
block used for training, that epoch is masked as a discontinuity rather than
scored by a model trained on it. This preserves causal integrity but reduces
available monitoring time.

V12B does **not** claim formal SPH=5 min/SOP=30 min. That experiment requires
new positive labels for −35 to −5 min and therefore a segmentation rebuild.
The existing −30 to −5 min analysis remains a 5-min minimum-warning analysis.


## 0–4. Validated V10/V11 loading, preprocessing, fitting, and alarm primitives


In [ ]:
# ==========================================================================
# 0. IMPORTS AND LOCKED V10 CONFIGURATION
# ==========================================================================
import gc
import json
import math
import re
import shutil
import time
import traceback
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import confusion_matrix, make_scorer, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    import xgboost
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError('V10 requires xgboost, which is normally preinstalled on Kaggle') from exc

RANDOM_SEED = 42
EPOCH_SEC = 10
SPH_SEC = 0
SOP_SEC = 30 * 60
REFRACTORY_SEC = 30 * 60
PRIMARY_ALARM_NAME = 'FP_5min_theta_0.70'
PRIMARY_FP_WINDOW_EPOCHS = 30
PRIMARY_FP_THRESHOLD = 0.70
N_SELECTED_FEATURES = 20
TRAIN_NEG_POS_RATIO = 1.0

MODEL_ORDER = ['RandomForest', 'XGBoost']
INNER_CV_MAX_SPLITS = 3
INNER_SCORING = 'MCC'
GRID_N_JOBS = 2

# F1-F4 retain V8's global Top-20 rule. Quotas apply only to mixed blocks.
BLOCK_BALANCED_QUOTAS = {
    ('Handcrafted', 'Graph_PLI'): {'Handcrafted': 10, 'Graph_PLI': 10},
    ('Handcrafted', 'Graph_wPLI'): {'Handcrafted': 10, 'Graph_wPLI': 10},
    ('Handcrafted', 'Graph_PLI', 'Graph_wPLI'): {
        'Handcrafted': 10, 'Graph_PLI': 5, 'Graph_wPLI': 5,
    },
}

ALARM_SENSITIVITY_SETTINGS = {
    'FP_2min_theta_0.70': {'mode': 'firing_power', 'window_epochs': 12, 'threshold': 0.70},
    PRIMARY_ALARM_NAME: {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.70},
    'FP_10min_theta_0.70': {'mode': 'firing_power', 'window_epochs': 60, 'threshold': 0.70},
    'FP_5min_theta_0.50': {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.50},
    'FP_5min_theta_0.80': {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.80},
    'Legacy_90_consecutive': {'mode': 'consecutive', 'k': 90},
}
assert ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME] == {
    'mode': 'firing_power',
    'window_epochs': PRIMARY_FP_WINDOW_EPOCHS,
    'threshold': PRIMARY_FP_THRESHOLD,
}

INPUT_SEARCH_ROOT = Path('/kaggle/input')
MODEL_READY_ROOT_OVERRIDE = None
TRAINING_TIME_ROOT_OVERRIDE = None  # V12C dataset root or parquet file
OUTPUT_ROOT = Path('/kaggle/working/MBEC_V12B_ROBUSTNESS')

# First run: True. To resume after interruption without deleting completed
# subjects, change to False and rerun the notebook from the beginning.
RESET_OUTPUT = False

RF_PARAMS = {
    'n_estimators': 500,
    'max_depth': 15,
    'min_samples_leaf': 4,
    'max_features': 'sqrt',
    'class_weight': None,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
}

XGB_BASE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'tree_method': 'hist',
    'n_jobs': 1,
}

# List-of-dictionaries prevents an unintended Cartesian product and keeps the
# nested computation feasible. Each dictionary is one pre-declared candidate.
XGB_PARAM_GRID = [
    {'n_estimators': [200], 'max_depth': [3], 'learning_rate': [0.05]},
    {'n_estimators': [400], 'max_depth': [3], 'learning_rate': [0.03]},
    {'n_estimators': [300], 'max_depth': [5], 'learning_rate': [0.05]},
    {'n_estimators': [200], 'max_depth': [5], 'learning_rate': [0.10]},
]

SVM_BASE_PARAMS = {
    'kernel': 'rbf',
    'class_weight': None,
    'probability': False,
    'cache_size': 1024,
}
SVM_PARAM_GRID = [
    {'C': [0.1], 'gamma': ['scale']},
    {'C': [1.0], 'gamma': ['scale']},
    {'C': [10.0], 'gamma': ['scale']},
    {'C': [1.0], 'gamma': [0.1]},
    {'C': [10.0], 'gamma': [0.01]},
]

BLOCK_PREFIX = {
    'Handcrafted': 'HC_',
    'Graph_PLI': 'PLI_',
    'Graph_wPLI': 'wPLI_',
}



# ==========================================================================
# 1. DISCOVER AND AUDIT THE MODEL-READY DATASET
# ==========================================================================
def discover_model_ready_root():
    if MODEL_READY_ROOT_OVERRIDE is not None:
        candidates = [Path(MODEL_READY_ROOT_OVERRIDE)]
    else:
        candidates = list(INPUT_SEARCH_ROOT.rglob('ModelReady_GAPAWARE_V4'))
    valid = []
    for root in candidates:
        required = [root / 'Metadata', root / 'Handcrafted', root / 'Graph_PLI',
                    root / 'Graph_wPLI', root / 'feature_set_definitions.json']
        if all(p.exists() for p in required):
            valid.append(root.resolve())
    valid = sorted(set(valid))
    if not valid:
        raise FileNotFoundError('Could not find a complete ModelReady_GAPAWARE_V4 dataset')
    if len(valid) > 1:
        raise RuntimeError(f'Multiple model-ready roots found: {valid}; set MODEL_READY_ROOT_OVERRIDE')
    return valid[0]


MODEL_ROOT = discover_model_ready_root()
with open(MODEL_ROOT / 'feature_set_definitions.json', encoding='utf-8') as file:
    FEATURE_SET_DEFINITIONS = json.load(file)

subjects = sorted(p.name for p in (MODEL_ROOT / 'Metadata').iterdir()
                  if p.is_dir() and re.fullmatch(r'chb\d{2}', p.name))
if not subjects:
    raise RuntimeError('No model-ready subjects found')

print('Model-ready root:', MODEL_ROOT)
print(f'Subjects ({len(subjects)}):', subjects)
print('Feature sets:', list(FEATURE_SET_DEFINITIONS))




# ==========================================================================
# 2. LOAD ONE SUBJECT AND ALIGN ALL THREE FEATURE BLOCKS
# ==========================================================================
def is_testing_stem(stem):
    return re.search(r'_block_\d+$', stem) is not None


def keys_for_metadata(meta):
    if 'Class' in meta.columns:
        return ['Patient', 'Class', 'SeizureID', 'Epoch']
    return ['Patient', 'SeizureID', 'Block', 'Epoch']


def feature_columns_for_blocks(frame, blocks):
    prefixes = tuple(BLOCK_PREFIX[b] for b in blocks)
    cols = [c for c in frame.columns if c.startswith(prefixes)]
    if not cols:
        raise ValueError(f'No features found for blocks={blocks}')
    return cols


def epoch_fingerprints(frame, fingerprint_cols):
    return pd.util.hash_pandas_object(
        frame[fingerprint_cols], index=False, categorize=False
    ).to_numpy(dtype=np.uint64)


def discover_training_timestamp_file():
    if TRAINING_TIME_ROOT_OVERRIDE is not None:
        candidate = Path(TRAINING_TIME_ROOT_OVERRIDE)
        candidates = ([candidate] if candidate.is_file() else
                      [candidate / 'training_epoch_timestamps.parquet'])
    else:
        candidates = list(INPUT_SEARCH_ROOT.rglob(
            'training_epoch_timestamps.parquet'))
    candidates = sorted(set(path.resolve() for path in candidates if path.exists()))
    if not candidates:
        raise FileNotFoundError(
            'V12C training_epoch_timestamps.parquet was not found. '
            'Attach the saved MBEC_V12C_TRAINING_TIMESTAMPS dataset.')
    if len(candidates) > 1:
        raise RuntimeError(
            f'Multiple V12C timestamp files found: {candidates}; '
            'set TRAINING_TIME_ROOT_OVERRIDE explicitly')
    return candidates[0]


_TRAINING_TIMESTAMP_FILE = None
_TRAINING_TIMESTAMP_CACHE = None


def load_training_timestamp_manifest():
    global _TRAINING_TIMESTAMP_FILE, _TRAINING_TIMESTAMP_CACHE
    if _TRAINING_TIMESTAMP_CACHE is None:
        _TRAINING_TIMESTAMP_FILE = discover_training_timestamp_file()
        manifest = pd.read_parquet(_TRAINING_TIMESTAMP_FILE)
        required = ['Patient', 'Class', 'SeizureID', 'Epoch', 'EpochStart']
        missing = [column for column in required if column not in manifest.columns]
        if missing:
            raise KeyError(f'V12C manifest missing columns: {missing}')
        manifest = manifest[required].copy()
        manifest['Patient'] = manifest['Patient'].astype(str)
        manifest['Class'] = manifest['Class'].replace(
            {'interictal': 0, 'preictal': 1})
        manifest['Class'] = pd.to_numeric(manifest['Class'], errors='raise').astype(int)
        manifest['SeizureID'] = manifest['SeizureID'].astype(int)
        manifest['Epoch'] = manifest['Epoch'].astype(int)
        manifest['EpochStart'] = pd.to_datetime(manifest['EpochStart'], utc=True)
        keys = ['Patient', 'Class', 'SeizureID', 'Epoch']
        if manifest.duplicated(keys).any():
            raise AssertionError('V12C timestamp manifest contains duplicate keys')
        if manifest['EpochStart'].isna().any():
            raise AssertionError('V12C timestamp manifest contains missing times')
        _TRAINING_TIMESTAMP_CACHE = manifest
        print('Loaded validated V12C timestamps:', _TRAINING_TIMESTAMP_FILE)
    return _TRAINING_TIMESTAMP_CACHE


def recover_training_epoch_starts(train_pool, stream_pool, subject):
    """Attach exact V12C training times using ModelReady epoch keys."""
    if 'EpochStart' in train_pool.columns:
        train_pool = train_pool.copy()
        train_pool['EpochStart'] = pd.to_datetime(train_pool['EpochStart'], utc=True)
        return train_pool

    manifest = load_training_timestamp_manifest()
    subject_times = manifest.loc[manifest['Patient'].eq(subject)].copy()
    if subject_times.empty:
        raise AssertionError(f'{subject}: no rows in V12C timestamp manifest')

    keys = ['Patient', 'Class', 'SeizureID', 'Epoch']
    result = train_pool.copy()
    result['Patient'] = result['Patient'].astype(str)
    result['Class'] = result['Class'].astype(int)
    result['SeizureID'] = result['SeizureID'].astype(int)
    result['Epoch'] = result['Epoch'].astype(int)
    result['_OriginalOrder'] = np.arange(len(result))
    result = result.merge(
        subject_times, on=keys, how='left', validate='one_to_one')
    result = result.sort_values('_OriginalOrder').drop(columns='_OriginalOrder')
    result = result.reset_index(drop=True)

    missing = int(result['EpochStart'].isna().sum())
    if missing:
        by_class = result.loc[result['EpochStart'].isna(), 'Class'].value_counts().to_dict()
        raise AssertionError(
            f'{subject}: V12C timestamp coverage failed for {missing} rows; '
            f'missing by class={by_class}')
    if len(result) != len(train_pool):
        raise AssertionError(f'{subject}: timestamp merge changed training row count')
    return result


def load_subject(subject, require_train_time=False):
    meta_files = sorted((MODEL_ROOT / 'Metadata' / subject).glob('*.parquet'))
    if not meta_files:
        raise FileNotFoundError(f'No metadata files for {subject}')
    training, streams = [], []

    for meta_path in meta_files:
        stem = meta_path.stem
        meta = pd.read_parquet(meta_path)
        keys = keys_for_metadata(meta)
        if meta.duplicated(keys).any():
            raise AssertionError(f'{subject}/{stem}: duplicate metadata epoch keys')
        frame = meta.copy()
        for block in BLOCK_PREFIX:
            path = MODEL_ROOT / block / subject / f'{stem}.parquet'
            if not path.exists():
                raise FileNotFoundError(path)
            features = pd.read_parquet(path)
            if features.duplicated(keys).any():
                raise AssertionError(f'{block}/{subject}/{stem}: duplicate keys')
            feature_cols = [c for c in features.columns if c not in keys]
            frame = frame.merge(features[keys + feature_cols], on=keys,
                                how='inner', validate='one_to_one')
        if len(frame) != len(meta):
            raise AssertionError(f'{subject}/{stem}: feature merge lost epochs')
        (streams if is_testing_stem(stem) else training).append(frame)

    train_pool = pd.concat(training, ignore_index=True)
    stream_pool = pd.concat(streams, ignore_index=True)
    train_pool['Class'] = train_pool['Class'].map({'interictal': 0, 'preictal': 1})
    if train_pool['Class'].isna().any():
        raise ValueError(f'{subject}: unknown training class')
    train_pool['Class'] = train_pool['Class'].astype(int)
    stream_pool['EpochStart'] = pd.to_datetime(stream_pool['EpochStart'], utc=True)
    if require_train_time:
        train_pool = recover_training_epoch_starts(train_pool, stream_pool, subject)

    # Strong cohort invariants inherited from segmentation.
    for sid, group in train_pool.groupby('SeizureID'):
        counts = group['Class'].value_counts().to_dict()
        if counts.get(1, 0) != 180 or counts.get(0, 0) != 540:
            raise AssertionError(f'{subject}/seizure {sid}: expected 180/540, got {counts}')
        target = stream_pool.loc[stream_pool['SeizureID'] == sid]
        if int(target['TargetPreictal'].sum()) != 180:
            raise AssertionError(f'{subject}/seizure {sid}: continuous preictal != 180 epochs')
        if target.duplicated(['Epoch']).any():
            raise AssertionError(f'{subject}/seizure {sid}: duplicate continuous Epoch')
    return train_pool, stream_pool




# ==========================================================================
# 3. TRAINING-ONLY BALANCING, SCALING, MI SELECTION AND INNER TUNING
# ==========================================================================
def balance_training(frame, fold_seed):
    positive = frame.loc[frame['Class'] == 1]
    negative = frame.loc[frame['Class'] == 0]
    if positive.empty or negative.empty:
        raise ValueError('Training fold must contain both classes')
    n_negative = min(len(negative), int(round(len(positive) * TRAIN_NEG_POS_RATIO)))
    negative = negative.sample(n=n_negative, replace=False, random_state=fold_seed)
    return (pd.concat([positive, negative], ignore_index=True)
              .sample(frac=1.0, random_state=fold_seed)
              .reset_index(drop=True))


def feature_block(feature_name):
    matches = [block for block, prefix in BLOCK_PREFIX.items()
               if feature_name.startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f'Cannot assign feature to exactly one block: {feature_name}')
    return matches[0]


def feature_selection_plan(blocks):
    block_set = set(blocks)
    for configured_blocks, quotas in BLOCK_BALANCED_QUOTAS.items():
        if len(blocks) == len(configured_blocks) and block_set == set(configured_blocks):
            if sum(quotas.values()) != N_SELECTED_FEATURES:
                raise AssertionError(f'Block quotas do not sum to {N_SELECTED_FEATURES}')
            return 'block_balanced_mi', dict(quotas)
    return 'global_top_mi', {'GLOBAL': N_SELECTED_FEATURES}


def select_feature_indices(feature_cols, mi_scores, blocks):
    mode, plan = feature_selection_plan(blocks)
    scores = np.asarray(mi_scores, dtype=float)
    if len(scores) != len(feature_cols):
        raise ValueError('MI-score/feature length mismatch')

    if mode == 'global_top_mi':
        k = min(N_SELECTED_FEATURES, len(feature_cols))
        selected_idx = np.sort(np.argsort(scores, kind='mergesort')[-k:])
        ranked = sorted(selected_idx, key=lambda i: (-scores[i], feature_cols[i]))
        records = []
        for rank, idx in enumerate(ranked, start=1):
            records.append({
                'Feature': feature_cols[idx], 'MIScore': float(scores[idx]),
                'FeatureBlock': feature_block(feature_cols[idx]),
                'RankWithinBlock': np.nan, 'RequestedBlockQuota': np.nan,
                'RankWithinSelected': rank,
            })
        return np.asarray(selected_idx, dtype=np.int64), records, mode, plan

    selected_idx, records = [], []
    for block in blocks:
        quota = int(plan[block])
        candidates = [i for i, name in enumerate(feature_cols)
                      if feature_block(name) == block]
        if len(candidates) < quota:
            raise ValueError(f'{block}: requested {quota} features, found {len(candidates)}')
        local = np.argsort(scores[candidates], kind='mergesort')[-quota:]
        chosen = [candidates[i] for i in local]
        ranked = sorted(chosen, key=lambda i: (-scores[i], feature_cols[i]))
        for block_rank, idx in enumerate(ranked, start=1):
            selected_idx.append(idx)
            records.append({
                'Feature': feature_cols[idx], 'MIScore': float(scores[idx]),
                'FeatureBlock': block, 'RankWithinBlock': block_rank,
                'RequestedBlockQuota': quota,
                'RankWithinSelected': len(records) + 1,
            })

    if len(selected_idx) != N_SELECTED_FEATURES or len(set(selected_idx)) != len(selected_idx):
        raise AssertionError('Block-balanced selector did not return 20 unique features')
    return (np.sort(np.asarray(selected_idx, dtype=np.int64)), records,
            mode, plan)


def prepare_fold(train, test, stream, feature_cols, blocks, fold_seed):
    balanced = balance_training(train, fold_seed)
    X_train = balanced[feature_cols].to_numpy(dtype=np.float32)
    y_train = balanced['Class'].to_numpy(dtype=np.int8)
    groups_train = balanced['SeizureID'].to_numpy(dtype=np.int32)
    X_test = test[feature_cols].to_numpy(dtype=np.float32)
    y_test = test['Class'].to_numpy(dtype=np.int8)
    X_stream = stream[feature_cols].to_numpy(dtype=np.float32)

    for name, array in [('train', X_train), ('test', X_test), ('stream', X_stream)]:
        if not np.isfinite(array).all():
            raise ValueError(f'{name}: NaN or Inf before modelling')

    scaler = StandardScaler()
    X_train_z = scaler.fit_transform(X_train)
    X_test_z = scaler.transform(X_test)
    X_stream_z = scaler.transform(X_stream)

    mi_scores = mutual_info_classif(X_train_z, y_train, random_state=RANDOM_SEED)
    selected_idx, selected, selection_mode, selection_plan = select_feature_indices(
        feature_cols, mi_scores, blocks)
    return {
        'X_train': X_train_z[:, selected_idx], 'y_train': y_train,
        'groups_train': groups_train,
        'X_test': X_test_z[:, selected_idx], 'y_test': y_test,
        'X_stream': X_stream_z[:, selected_idx],
        'selected': selected, 'balanced_n': len(balanced),
        'scaler': scaler, 'selected_idx': selected_idx,
        'selection_mode': selection_mode, 'selection_plan': selection_plan,
    }


def _jsonable_params(params):
    clean = {}
    for key, value in params.items():
        clean[key] = value.item() if isinstance(value, np.generic) else value
    return clean


def fit_classifier(model_name, X, y, groups, fold_seed):
    start = time.perf_counter()
    if model_name == 'RandomForest':
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X, y)
        summary = {
            'TuningMode': 'locked_V9A_parameters', 'InnerSplits': 0,
            'BestInnerMCC': np.nan, 'BestInnerMCC_SD': np.nan,
            'BestParams': json.dumps(_jsonable_params(RF_PARAMS), sort_keys=True),
        }
        candidates = []
    else:
        unique_groups = np.unique(groups)
        n_splits = min(INNER_CV_MAX_SPLITS, len(unique_groups))
        if n_splits < 2:
            raise ValueError('Grouped inner CV requires at least two training seizures')
        cv = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=fold_seed)
        scorer = make_scorer(matthews_corrcoef)
        if model_name == 'XGBoost':
            estimator = XGBClassifier(**XGB_BASE_PARAMS)
            grid = XGB_PARAM_GRID
        elif model_name == 'SVM_RBF':
            estimator = SVC(**SVM_BASE_PARAMS)
            grid = SVM_PARAM_GRID
        else:
            raise ValueError(f'Unknown model: {model_name}')
        search = GridSearchCV(
            estimator=estimator, param_grid=grid, scoring=scorer, cv=cv,
            refit=True, n_jobs=GRID_N_JOBS, error_score='raise',
            return_train_score=False)
        search.fit(X, y, groups=groups)
        model = search.best_estimator_
        best_index = int(search.best_index_)
        summary = {
            'TuningMode': 'grouped_inner_CV', 'InnerSplits': n_splits,
            'BestInnerMCC': float(search.cv_results_['mean_test_score'][best_index]),
            'BestInnerMCC_SD': float(search.cv_results_['std_test_score'][best_index]),
            'BestParams': json.dumps(_jsonable_params(search.best_params_), sort_keys=True),
        }
        candidates = []
        for idx, params in enumerate(search.cv_results_['params']):
            candidates.append({
                'CandidateIndex': idx,
                'Params': json.dumps(_jsonable_params(params), sort_keys=True),
                'MeanInnerMCC': float(search.cv_results_['mean_test_score'][idx]),
                'SDInnerMCC': float(search.cv_results_['std_test_score'][idx]),
                'RankInnerMCC': int(search.cv_results_['rank_test_score'][idx]),
                'Selected': int(idx == best_index),
            })
    summary['FitAndTuneTime_sec'] = time.perf_counter() - start
    return model, summary, candidates


def build_interictal_owner_map(train_pool, fingerprint_cols):
    negative = train_pool.loc[train_pool['Class'] == 0]
    hashes = epoch_fingerprints(negative, fingerprint_cols)
    owners = negative['SeizureID'].to_numpy(dtype=int)
    mapping = pd.DataFrame({'Hash': hashes, 'Owner': owners})
    conflicts = mapping.groupby('Hash')['Owner'].nunique()
    if (conflicts > 1).any():
        raise AssertionError('An interictal fingerprint belongs to multiple LOSO folds')
    return mapping.drop_duplicates('Hash').set_index('Hash')['Owner'].to_dict()


def predict_pipeline(frame, feature_cols, fitted):
    scaler, selected_idx, model = fitted
    values = frame[feature_cols].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all():
        raise ValueError('NaN or Inf during cross-fitted stream prediction')
    transformed = scaler.transform(values)[:, selected_idx]
    return model.predict(transformed).astype(np.int8)



# ==========================================================================
# 4. CAUSAL CONTINUOUS ALARM, FAR/H AND TIME-TO-WARNING
# ==========================================================================
def _validate_alarm_setting(setting):
    mode = setting.get('mode')
    if mode == 'firing_power':
        window = int(setting['window_epochs'])
        threshold = float(setting['threshold'])
        if window < 1 or not (0 < threshold <= 1):
            raise ValueError(f'Invalid firing-power setting: {setting}')
        return mode, window, threshold, math.ceil(window * threshold)
    if mode == 'consecutive':
        k = int(setting['k'])
        if k < 1:
            raise ValueError(f'Invalid consecutive setting: {setting}')
        return mode, k, 1.0, k
    raise ValueError(f'Unknown alarm mode: {mode}')


def run_causal_alarm(y_pred, stream, setting):
    stream = stream.reset_index(drop=True)
    predictions = np.asarray(y_pred, dtype=np.int8)
    if len(predictions) != len(stream):
        raise ValueError('Prediction/stream length mismatch')
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError('Alarm input predictions must be binary')

    mode, window_epochs, threshold, required_positive = _validate_alarm_setting(setting)

    history = deque(maxlen=window_epochs)
    refractory_until = None
    previous_start = None
    alarms = []

    for idx, (pred, row) in enumerate(zip(predictions, stream.itertuples(index=False))):
        start = row.EpochStart
        end = start + pd.Timedelta(seconds=EPOCH_SEC)
        gap = bool(row.Discontinuity)
        if previous_start is not None:
            elapsed = (start - previous_start).total_seconds()
            gap = gap or elapsed > EPOCH_SEC + 0.5 or elapsed <= 0
        previous_start = start

        phase = str(row.Phase).lower()
        valid_state = phase in {'interictal', 'preictal'}
        if gap or not valid_state:
            history.clear()
            continue
        if refractory_until is not None and end < refractory_until:
            history.clear()
            continue

        if mode == 'consecutive':
            if pred == 0:
                history.clear()
            else:
                history.append(1)
        else:
            history.append(int(pred))

        full_window = len(history) == window_epochs
        positive_count = int(sum(history))
        if full_window and positive_count >= required_positive:
            alarms.append({
                'Index': idx,
                'AlarmTime': end,
                'Phase': phase,
                'EligibleForFAR': int(row.EligibleForFAR),
                'TargetPreictal': int(row.TargetPreictal),
                'WindowEpochs': window_epochs,
                'PositiveEpochs': positive_count,
                'FiringPower': positive_count / window_epochs,
            })
            history.clear()
            refractory_until = end + pd.Timedelta(seconds=REFRACTORY_SEC)
    return alarms


def evaluate_alarm(y_pred, stream, setting):
    stream = stream.sort_values('Epoch', kind='stable').reset_index(drop=True)
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if len(target) != 180:
        raise AssertionError(f'Expected 180 target-preictal epochs, got {len(target)}')
    seizure_onset = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    valid_start = seizure_onset - pd.Timedelta(seconds=SPH_SEC + SOP_SEC)
    valid_end = seizure_onset - pd.Timedelta(seconds=SPH_SEC)

    alarms = run_causal_alarm(y_pred, stream, setting)
    true_alarms = [a for a in alarms if valid_start <= a['AlarmTime'] < valid_end]
    late_alarms = [a for a in alarms if valid_end <= a['AlarmTime'] < seizure_onset]
    false_alarms = [a for a in alarms if a['EligibleForFAR'] == 1]
    eligible_hours = float(stream['EligibleForFAR'].astype(bool).sum() * EPOCH_SEC / 3600.0)
    far_h = len(false_alarms) / eligible_hours if eligible_hours > 0 else np.nan
    ttw = ((seizure_onset - true_alarms[0]['AlarmTime']).total_seconds() / 60.0
           if true_alarms else np.nan)
    return {
        'SeizurePredicted': int(bool(true_alarms)),
        'FalseAlarms': len(false_alarms),
        'LateAlarms': len(late_alarms),
        'EligibleInterictalHours': eligible_hours,
        'FAR_per_h': far_h,
        'TimeToWarning_min': ttw,
        'TotalAlarms': len(alarms),
    }


def run_alarm_unit_tests():
    onset = pd.Timestamp('2076-01-01T01:00:00Z')
    starts = pd.date_range(onset - pd.Timedelta(minutes=40), periods=240, freq='10s')
    stream = pd.DataFrame({
        'Epoch': np.arange(240), 'EpochStart': starts,
        'Phase': ['interictal'] * 60 + ['preictal'] * 180,
        'TargetPreictal': [0] * 60 + [1] * 180,
        'EligibleForFAR': [1] * 60 + [0] * 180,
        'Discontinuity': 0,
    })
    predictions = np.array([0] * 60 + [1] * 180)
    primary = ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME]
    result = evaluate_alarm(predictions, stream, primary)
    assert result['SeizurePredicted'] == 1
    # The 5-min causal window already contains nine preceding interictal
    # negatives, so the threshold is crossed after 21 positive epochs (3.5 min).
    assert abs(result['TimeToWarning_min'] - 26.5) < 1e-9
    assert result['FalseAlarms'] == 0

    # Nine isolated negatives are tolerated: 21/30 still reaches theta=0.70.
    noisy = predictions.copy()
    noisy[61:70] = 0
    result = evaluate_alarm(noisy, stream, primary)
    assert result['SeizurePredicted'] == 1
    assert abs(result['TimeToWarning_min'] - 25.0) < 1e-9

    # A discontinuity clears all prior evidence. With only 20 valid epochs
    # after the gap, no full 30-epoch firing-power window can be evaluated.
    broken = stream.copy()
    short_predictions = np.zeros_like(predictions)
    short_predictions[60:90] = 1
    broken.loc[70, 'Discontinuity'] = 1
    result = evaluate_alarm(short_predictions, broken, primary)
    assert result['SeizurePredicted'] == 0

    # Legacy rule remains reproducible only as a sensitivity analysis.
    result = evaluate_alarm(
        predictions, stream, ALARM_SENSITIVITY_SETTINGS['Legacy_90_consecutive'])
    assert result['SeizurePredicted'] == 1
    assert abs(result['TimeToWarning_min'] - 15.0) < 1e-9
    print('Causal alarm unit tests: PASS')


run_alarm_unit_tests()


## 5. Online-safe alarm generator inherited from V11B


In [ ]:
def run_online_safe_alarm(y_pred, stream, setting):
    """Generate alarms without using Phase, TargetPreictal, or EligibleForFAR.

    Only timestamp discontinuities and the fixed refractory rule affect alarm
    generation. Retrospective labels are attached to emitted alarms solely for
    subsequent scoring.
    """
    stream = stream.reset_index(drop=True)
    predictions = np.asarray(y_pred, dtype=np.int8)
    if len(predictions) != len(stream):
        raise ValueError('Prediction/stream length mismatch')
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError('Alarm input predictions must be binary')

    mode, window_epochs, threshold, required_positive = _validate_alarm_setting(setting)
    history = deque(maxlen=window_epochs)
    refractory_until = None
    previous_start = None
    alarms = []

    for idx, (pred, row) in enumerate(zip(predictions, stream.itertuples(index=False))):
        start = row.EpochStart
        end = start + pd.Timedelta(seconds=EPOCH_SEC)
        gap = bool(row.Discontinuity)
        if previous_start is not None:
            elapsed = (start - previous_start).total_seconds()
            gap = gap or elapsed > EPOCH_SEC + 0.5 or elapsed <= 0
        previous_start = start

        if gap:
            history.clear()
            continue
        if refractory_until is not None and end < refractory_until:
            history.clear()
            continue

        if mode == 'consecutive':
            if pred == 0:
                history.clear()
            else:
                history.append(1)
        else:
            history.append(int(pred))

        positive_count = int(sum(history))
        if len(history) == window_epochs and positive_count >= required_positive:
            alarms.append({
                'Index': idx,
                'AlarmTime': end,
                'Phase': str(row.Phase).lower(),
                'EligibleForFAR': int(row.EligibleForFAR),
                'TargetPreictal': int(row.TargetPreictal),
                'WindowEpochs': window_epochs,
                'PositiveEpochs': positive_count,
                'FiringPower': positive_count / window_epochs,
            })
            history.clear()
            refractory_until = end + pd.Timedelta(seconds=REFRACTORY_SEC)
    return alarms


def evaluate_alarm_generator(y_pred, stream, setting, generation_mode):
    stream = stream.sort_values('Epoch', kind='stable').reset_index(drop=True)
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if len(target) != 180:
        raise AssertionError(f'Expected 180 target-preictal epochs, got {len(target)}')
    seizure_onset = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    valid_start = seizure_onset - pd.Timedelta(seconds=SPH_SEC + SOP_SEC)
    valid_end = seizure_onset - pd.Timedelta(seconds=SPH_SEC)

    if generation_mode == 'label_gated_v10':
        alarms = run_causal_alarm(y_pred, stream, setting)
    elif generation_mode == 'online_safe_gap_only':
        alarms = run_online_safe_alarm(y_pred, stream, setting)
    else:
        raise ValueError(generation_mode)

    true_alarms = [a for a in alarms if valid_start <= a['AlarmTime'] < valid_end]
    late_alarms = [a for a in alarms if valid_end <= a['AlarmTime'] < seizure_onset]
    false_alarms = [a for a in alarms if a['EligibleForFAR'] == 1]
    eligible_hours = float(stream['EligibleForFAR'].astype(bool).sum() * EPOCH_SEC / 3600.0)
    return {
        'AlarmGenerationMode': generation_mode,
        'SeizurePredicted': int(bool(true_alarms)),
        'FalseAlarms': len(false_alarms),
        'LateAlarms': len(late_alarms),
        'EligibleInterictalHours': eligible_hours,
        'FAR_per_h': len(false_alarms)/eligible_hours if eligible_hours > 0 else np.nan,
        'TimeToWarning_min': (
            (seizure_onset-true_alarms[0]['AlarmTime']).total_seconds()/60
            if true_alarms else np.nan),
        'TotalAlarms': len(alarms),
    }


def run_online_safe_alarm_unit_tests():
    onset = pd.Timestamp('2076-01-01T01:00:00Z')
    starts = pd.date_range(onset-pd.Timedelta(minutes=40), periods=240, freq='10s')
    stream = pd.DataFrame({
        'Epoch': np.arange(240), 'EpochStart': starts,
        'Phase': ['excluded']*30 + ['interictal']*30 + ['preictal']*180,
        'TargetPreictal': [0]*60 + [1]*180,
        'EligibleForFAR': [0]*30 + [1]*30 + [0]*180,
        'Discontinuity': 0,
    })
    pred = np.array([0]*60 + [1]*180)
    result = evaluate_alarm_generator(
        pred, stream, ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME],
        'online_safe_gap_only')
    assert result['SeizurePredicted'] == 1
    broken = stream.copy()
    broken.loc[70, 'Discontinuity'] = 1
    short = np.zeros_like(pred); short[60:90] = 1
    result = evaluate_alarm_generator(
        short, broken, ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME],
        'online_safe_gap_only')
    assert result['SeizurePredicted'] == 0
    print('Online-safe alarm unit tests: PASS')

run_online_safe_alarm_unit_tests()


## 6. V12B stage configuration


In [ ]:
# ============================================================================
# V12B STAGE CONFIGURATION
# ============================================================================
RUN_ALL21 = False
RUN_GUARD10 = True
RUN_FORWARD = True
RUN_NO_GAMMA = False
RUN_MULTI_SEED = False

# Change individual switches to False when splitting work across Kaggle sessions.
SAVE_ALL21_STREAM_PREDICTIONS = True
GUARD_MIN = 10
MIN_PRIOR_SEIZURES = 2
MULTI_SEEDS_EXTRA = [43, 44, 45, 46]  # seed 42 is reused from all21

ALL_MODELS = ['RandomForest', 'XGBoost', 'SVM_RBF']
ALL_FEATURES = list(FEATURE_SET_DEFINITIONS)
ALL21_PAIRS = [(m, f) for m in ALL_MODELS for f in ALL_FEATURES]
FOCAL_PAIRS = [
    ('RandomForest', 'F1_Handcrafted'),
    ('RandomForest', 'F7_All'),
    ('XGBoost', 'F1_Handcrafted'),
    ('XGBoost', 'F6_Handcrafted_wPLI'),
]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'checkpoints').mkdir(exist_ok=True)
V12B_CHECKPOINT_RE = re.compile(
    r'^(all21|guard10|forward|nogamma|multiseed)__seed_\d{3}__chb\d{2}\.csv$')
(OUTPUT_ROOT / 'stream_predictions').mkdir(exist_ok=True)

# Resume helper: import checkpoint CSVs from any previously attached V12B
# Kaggle output dataset. Large stream parquets are intentionally not copied.
IMPORT_PRIOR_CHECKPOINTS = True
if IMPORT_PRIOR_CHECKPOINTS and INPUT_SEARCH_ROOT.exists():
    imported = 0
    for prior_dir in INPUT_SEARCH_ROOT.rglob('checkpoints'):
        if prior_dir.resolve() == (OUTPUT_ROOT / 'checkpoints').resolve():
            continue
        for source in prior_dir.glob('*.csv'):
            if V12B_CHECKPOINT_RE.fullmatch(source.name) is None:
                continue
            destination = OUTPUT_ROOT / 'checkpoints' / source.name
            if not destination.exists():
                shutil.copy2(source, destination)
                imported += 1
    print('Imported prior V12B checkpoints:', imported)

print('Output:', OUTPUT_ROOT)
print('All-configuration pairs:', len(ALL21_PAIRS))
print('Focal robustness pairs:', FOCAL_PAIRS)
print('Stages:', {
    'all21': RUN_ALL21, 'guard10': RUN_GUARD10, 'forward': RUN_FORWARD,
    'nogamma': RUN_NO_GAMMA, 'multiseed': RUN_MULTI_SEED,
})


## 7. V12B fitting, guard-gap, scoring, and export helpers


In [ ]:
def fit_classifier_v12(model_name, X, y, groups, fold_seed, model_seed):
    """Fit with V10/V11-compatible seed semantics.

    fold_seed controls training subsampling and grouped inner-CV splitting.
    model_seed is fixed within a replicate (42 for the primary analysis),
    matching V10/V11; it changes only across the multiseed replicates.
    """
    start = time.perf_counter()
    if model_name == 'RandomForest':
        params = dict(RF_PARAMS)
        params['random_state'] = int(model_seed)
        model = RandomForestClassifier(**params)
        model.fit(X, y)
        summary = {
            'TuningMode': 'locked_V9A_parameters', 'InnerSplits': 0,
            'BestInnerMCC': np.nan, 'BestInnerMCC_SD': np.nan,
            'BestParams': json.dumps(_jsonable_params(params), sort_keys=True),
        }
        candidates = []
    else:
        unique_groups = np.unique(groups)
        n_splits = min(INNER_CV_MAX_SPLITS, len(unique_groups))
        if n_splits < 2:
            raise ValueError('Grouped inner CV requires at least two training seizures')
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                  random_state=int(fold_seed))
        scorer = make_scorer(matthews_corrcoef)
        if model_name == 'XGBoost':
            params = dict(XGB_BASE_PARAMS)
            params['random_state'] = int(model_seed)
            estimator = XGBClassifier(**params)
            grid = XGB_PARAM_GRID
        elif model_name == 'SVM_RBF':
            estimator = SVC(**SVM_BASE_PARAMS)
            grid = SVM_PARAM_GRID
        else:
            raise ValueError(model_name)
        search = GridSearchCV(
            estimator=estimator, param_grid=grid, scoring=scorer, cv=cv,
            refit=True, n_jobs=GRID_N_JOBS, error_score='raise',
            return_train_score=False)
        search.fit(X, y, groups=groups)
        model = search.best_estimator_
        best = int(search.best_index_)
        summary = {
            'TuningMode': 'grouped_inner_CV', 'InnerSplits': n_splits,
            'BestInnerMCC': float(search.cv_results_['mean_test_score'][best]),
            'BestInnerMCC_SD': float(search.cv_results_['std_test_score'][best]),
            'BestParams': json.dumps(_jsonable_params(search.best_params_), sort_keys=True),
        }
        candidates = []
    summary['FoldSeed'] = int(fold_seed)
    summary['ModelSeed'] = int(model_seed)
    summary['FitAndTuneTime_sec'] = time.perf_counter() - start
    return model, summary, candidates


def epoch_counts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        np.asarray(y_true), np.asarray(y_pred), labels=[0, 1]).ravel()
    return int(tn), int(fp), int(fn), int(tp)


def is_gamma_feature(name):
    return re.search(r'(^|_)gamma(_|$)', str(name).lower()) is not None


def feature_columns_v12(frame, blocks, drop_gamma=False):
    columns = feature_columns_for_blocks(frame, blocks)
    if drop_gamma:
        columns = [c for c in columns if not is_gamma_feature(c)]
    if not columns:
        raise ValueError(f'No features remain for blocks={blocks}, drop_gamma={drop_gamma}')
    return columns


def temporal_guard_filter(train, test, guard_min):
    """Remove training interictal epochs near held-out interictal timestamps."""
    if guard_min <= 0:
        return train.copy(), 0
    test_times = (test.loc[test['Class'].eq(0), 'EpochStart']
                  .astype('int64').sort_values().to_numpy())
    if len(test_times) == 0:
        return train.copy(), 0
    negative = train['Class'].eq(0).to_numpy()
    train_times = train['EpochStart'].astype('int64').to_numpy()
    positions = np.searchsorted(test_times, train_times)
    max_ns = int(pd.Timedelta(minutes=guard_min).value)
    near = np.zeros(len(train), dtype=bool)
    left = positions > 0
    near[left] |= np.abs(train_times[left] - test_times[positions[left]-1]) <= max_ns
    right = positions < len(test_times)
    near[right] |= np.abs(train_times[right] - test_times[positions[right]]) <= max_ns
    remove = negative & near
    filtered = train.loc[~remove].copy()
    if filtered['Class'].nunique() != 2:
        raise ValueError('Guard gap removed one training class')
    return filtered, int(remove.sum())


def score_online_v12(y_pred, stream):
    stream = stream.sort_values('Epoch', kind='stable').reset_index(drop=True)
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if len(target) != 180:
        raise AssertionError(f'Expected 180 target-preictal epochs, got {len(target)}')
    onset = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    valid_start = onset - pd.Timedelta(minutes=30)
    mwt5_end = onset - pd.Timedelta(minutes=5)
    alarms = run_online_safe_alarm(
        y_pred, stream, ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME])
    true0 = [a for a in alarms if valid_start <= a['AlarmTime'] < onset]
    true5 = [a for a in alarms if valid_start <= a['AlarmTime'] < mwt5_end]
    false = [a for a in alarms if a['EligibleForFAR'] == 1]
    hours = float(stream['EligibleForFAR'].astype(bool).sum() * EPOCH_SEC / 3600)
    return {
        'SeizurePredicted_SPH0': int(bool(true0)),
        'SeizurePredicted_MWT5': int(bool(true5)),
        'DetectionOnlyFinal5min': int(bool(true0) and not bool(true5)),
        'FalseAlarms': len(false),
        'EligibleInterictalHours': hours,
        'FAR_per_h': len(false)/hours if hours else np.nan,
        'TTW_SPH0_min': ((onset-true0[0]['AlarmTime']).total_seconds()/60
                         if true0 else np.nan),
        'TTW_MWT5_min': ((onset-true5[0]['AlarmTime']).total_seconds()/60
                         if true5 else np.nan),
        'TotalAlarms': len(alarms),
        'FoldStreamHours': len(stream) * EPOCH_SEC / 3600,
    }


def save_prediction_v12(stage, seed, subject, model_name, feature_set, sid,
                        stream, y_stream, owners):
    out = stream[[
        'Epoch', 'EpochStart', 'Phase', 'TargetPreictal',
        'EligibleForFAR', 'Discontinuity'
    ]].copy()
    out.insert(0, 'HeldOutSeizure', int(sid))
    out.insert(0, 'FeatureSet', feature_set)
    out.insert(0, 'Model', model_name)
    out.insert(0, 'Subject', subject)
    out.insert(0, 'Seed', int(seed))
    out.insert(0, 'Stage', stage)
    out['Prediction'] = np.asarray(y_stream, dtype=np.int8)
    out['EpochOwnerSeizure'] = np.asarray(owners, dtype=np.int32)
    folder = OUTPUT_ROOT / 'stream_predictions' / stage / subject
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{model_name}__{feature_set}__seizure_{int(sid):03d}.parquet'
    out.to_parquet(path, index=False, compression='zstd')
    return path


def checkpoint_path(stage, seed, subject):
    return OUTPUT_ROOT / 'checkpoints' / f'{stage}__seed_{int(seed):03d}__{subject}.csv'


## 8. Standard LOSO stages: all21, guard-gap, no-gamma, and extra seeds


In [ ]:
def run_standard_subject(subject, stage, pairs, seed, guard_min=0,
                         drop_gamma=False, save_predictions=False):
    train_pool, stream_pool = load_subject(
        subject, require_train_time=(guard_min > 0))
    seizure_ids = sorted(train_pool.loc[train_pool['Class'].eq(1), 'SeizureID'].unique())
    fingerprint_cols = feature_columns_for_blocks(train_pool, ['Handcrafted'])
    owner_map = build_interictal_owner_map(train_pool, fingerprint_cols)

    streams, owners_by_sid, qc_by_sid = {}, {}, {}
    for sid in map(int, seizure_ids):
        stream = (stream_pool.loc[stream_pool['SeizureID'].eq(sid)]
                  .sort_values('Epoch', kind='stable').reset_index(drop=True))
        hashes = epoch_fingerprints(stream, fingerprint_cols)
        eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
        owners = np.full(len(stream), -1, dtype=np.int32)
        if eligible.any():
            owners[eligible] = np.array(
                [owner_map.get(int(h), -1) for h in hashes[eligible]], dtype=np.int32)
        streams[sid] = stream
        owners_by_sid[sid] = owners
        qc_by_sid[sid] = {
            'StreamEpochs': len(stream),
            'EligibleInterictalEpochs': int(eligible.sum()),
            'CrossFitMatchedInterictalEpochs': int((eligible & (owners >= 0)).sum()),
        }

    pairs_by_feature = {}
    for model, feature in pairs:
        pairs_by_feature.setdefault(feature, []).append(model)

    rows = []
    for feature_set, models in pairs_by_feature.items():
        blocks = FEATURE_SET_DEFINITIONS[feature_set]
        feature_cols = feature_columns_v12(train_pool, blocks, drop_gamma=drop_gamma)
        caches = {model: {} for model in models}
        guard_removed = {}
        print(stage, seed, subject, feature_set, models, len(feature_cols), flush=True)

        for fold_index, sid in enumerate(map(int, seizure_ids)):
            train = train_pool.loc[train_pool['SeizureID'].ne(sid)].copy()
            test = (train_pool.loc[train_pool['SeizureID'].eq(sid)]
                    .sort_values(['Class', 'Epoch'], kind='stable').copy())
            train, removed = temporal_guard_filter(train, test, guard_min)
            guard_removed[sid] = removed
            fold_seed = int(seed + fold_index)
            prepared = prepare_fold(train, test, streams[sid], feature_cols, blocks, fold_seed)
            for model_name in models:
                model, tuning, _ = fit_classifier_v12(
                    model_name, prepared['X_train'], prepared['y_train'],
                    prepared['groups_train'], fold_seed, int(seed))
                caches[model_name][sid] = {
                    'fitted': (prepared['scaler'], prepared['selected_idx'], model),
                    'y_test': prepared['y_test'],
                    'y_test_pred': model.predict(prepared['X_test']).astype(np.int8),
                    'default_stream_pred': model.predict(prepared['X_stream']).astype(np.int8),
                    'tuning': tuning,
                    'balanced_n': prepared['balanced_n'],
                }

        for model_name in models:
            cache = caches[model_name]
            for sid in map(int, seizure_ids):
                stream = streams[sid]
                owners = owners_by_sid[sid]
                eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
                result = cache[sid]
                y_stream = result['default_stream_pred'].copy()
                for owner in sorted(set(owners[eligible]) - {-1}):
                    idx = np.flatnonzero(eligible & (owners == owner))
                    y_stream[idx] = predict_pipeline(
                        stream.iloc[idx], feature_cols, cache[int(owner)]['fitted'])
                if save_predictions:
                    save_prediction_v12(stage, seed, subject, model_name,
                                        feature_set, sid, stream, y_stream, owners)
                tn, fp, fn, tp = epoch_counts(result['y_test'], result['y_test_pred'])
                alarm = score_online_v12(y_stream, stream)
                rows.append({
                    'Stage': stage, 'Seed': int(seed), 'Subject': subject,
                    'Model': model_name, 'FeatureSet': feature_set,
                    'HeldOutSeizure': int(sid), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                    'GuardMinutes': int(guard_min),
                    'GuardRemovedTrainingInterictalEpochs': int(guard_removed[sid]),
                    'GammaRemoved': int(drop_gamma),
                    'RawFeaturesAfterFilter': len(feature_cols),
                    'BalancedTrainingEpochs': result['balanced_n'],
                    **qc_by_sid[sid], **result['tuning'], **alarm,
                })
        del caches
        gc.collect()
    return pd.DataFrame(rows)


## 9. Chronological forward-chaining stage


In [ ]:
def run_forward_subject(subject, pairs, seed=RANDOM_SEED):
    train_pool, stream_pool = load_subject(subject, require_train_time=True)
    seizure_ids = sorted(train_pool.loc[train_pool['Class'].eq(1), 'SeizureID'].unique())
    onset = {}
    for sid in map(int, seizure_ids):
        target = stream_pool.loc[
            stream_pool['SeizureID'].eq(sid) & stream_pool['TargetPreictal'].astype(bool)]
        onset[sid] = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    chronological = sorted(map(int, seizure_ids), key=lambda sid: onset[sid])

    fingerprint_cols = feature_columns_for_blocks(train_pool, ['Handcrafted'])
    owner_map = build_interictal_owner_map(train_pool, fingerprint_cols)
    pairs_by_feature = {}
    for model, feature in pairs:
        pairs_by_feature.setdefault(feature, []).append(model)

    rows = []
    for feature_set, models in pairs_by_feature.items():
        blocks = FEATURE_SET_DEFINITIONS[feature_set]
        feature_cols = feature_columns_v12(train_pool, blocks, drop_gamma=False)
        for target_index, sid in enumerate(chronological):
            prior = chronological[:target_index]
            if len(prior) < MIN_PRIOR_SEIZURES:
                continue
            target_onset = onset[sid]
            train_candidate = train_pool.loc[train_pool['SeizureID'].isin(prior)].copy()
            test_candidate = train_pool.loc[train_pool['SeizureID'].eq(sid)].copy()
            stream_candidate = stream_pool.loc[stream_pool['SeizureID'].eq(sid)].copy()
            # Seizure-ID ordering alone is insufficient because an assigned
            # interictal chunk can occur later in clock time. Enforce timestamp
            # chronology on training, classification test, and event stream.
            train = train_candidate.loc[train_candidate['EpochStart'] < target_onset].copy()
            test = (test_candidate.loc[test_candidate['EpochStart'] < target_onset]
                    .sort_values(['Class', 'Epoch'], kind='stable').copy())
            stream = (stream_candidate.loc[stream_candidate['EpochStart'] < target_onset]
                      .sort_values('Epoch', kind='stable').reset_index(drop=True).copy())
            removed_train_chronology = len(train_candidate) - len(train)
            removed_test_chronology = len(test_candidate) - len(test)
            removed_stream_chronology = len(stream_candidate) - len(stream)
            hashes = epoch_fingerprints(stream, fingerprint_cols)
            eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
            owners = np.full(len(stream), -1, dtype=np.int32)
            if eligible.any():
                owners[eligible] = np.array(
                    [owner_map.get(int(h), -1) for h in hashes[eligible]], dtype=np.int32)
            blocked = eligible & np.isin(owners, np.asarray(prior, dtype=int))
            # A blocked row is skipped by the causal generator and creates a reset.
            stream.loc[blocked, 'EligibleForFAR'] = 0
            stream.loc[blocked, 'Discontinuity'] = 1

            fold_seed = int(seed + target_index)
            prepared = prepare_fold(train, test, stream, feature_cols, blocks, fold_seed)
            for model_name in models:
                model, tuning, _ = fit_classifier_v12(
                    model_name, prepared['X_train'], prepared['y_train'],
                    prepared['groups_train'], fold_seed, int(seed))
                y_test_pred = model.predict(prepared['X_test']).astype(np.int8)
                y_stream = model.predict(prepared['X_stream']).astype(np.int8)
                tn, fp, fn, tp = epoch_counts(prepared['y_test'], y_test_pred)
                alarm = score_online_v12(y_stream, stream)
                rows.append({
                    'Stage': 'forward', 'Seed': int(seed), 'Subject': subject,
                    'Model': model_name, 'FeatureSet': feature_set,
                    'HeldOutSeizure': int(sid), 'ChronologicalIndex': target_index,
                    'TrainingSeizures': len(prior),
                    'FutureSeizuresExcluded': len(chronological)-target_index-1,
                    'ChronologyRemovedTrainingEpochs': removed_train_chronology,
                    'ChronologyRemovedTestEpochs': removed_test_chronology,
                    'ChronologyRemovedStreamEpochs': removed_stream_chronology,
                    'RemovedOverlappingStreamEpochs': int(blocked.sum()),
                    'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                    'GuardMinutes': 0, 'GammaRemoved': 0,
                    'RawFeaturesAfterFilter': len(feature_cols),
                    'BalancedTrainingEpochs': prepared['balanced_n'],
                    **tuning, **alarm,
                })
    return pd.DataFrame(rows)


## 10. Resume-safe stage execution


In [ ]:
def execute_standard_stage(stage, pairs, seeds, guard_min=0,
                           drop_gamma=False, save_predictions=False):
    for seed in seeds:
        for subject in subjects:
            path = checkpoint_path(stage, seed, subject)
            if path.exists():
                print('Skip checkpoint:', path.name)
                continue
            try:
                result = run_standard_subject(
                    subject, stage, pairs, seed, guard_min=guard_min,
                    drop_gamma=drop_gamma, save_predictions=save_predictions)
                result.to_csv(path, index=False)
                print('Saved:', path.name, len(result), 'rows')
            except Exception:
                print('FAILED:', stage, seed, subject)
                traceback.print_exc()
                raise


if RUN_ALL21:
    execute_standard_stage(
        'all21', ALL21_PAIRS, [RANDOM_SEED],
        save_predictions=SAVE_ALL21_STREAM_PREDICTIONS)

if RUN_GUARD10:
    execute_standard_stage(
        'guard10', FOCAL_PAIRS, [RANDOM_SEED], guard_min=GUARD_MIN)

if RUN_NO_GAMMA:
    execute_standard_stage(
        'nogamma', FOCAL_PAIRS, [RANDOM_SEED], drop_gamma=True)

if RUN_MULTI_SEED:
    execute_standard_stage(
        'multiseed', FOCAL_PAIRS, MULTI_SEEDS_EXTRA)

if RUN_FORWARD:
    for subject in subjects:
        path = checkpoint_path('forward', RANDOM_SEED, subject)
        if path.exists():
            print('Skip checkpoint:', path.name)
            continue
        try:
            result = run_forward_subject(subject, FOCAL_PAIRS, RANDOM_SEED)
            result.to_csv(path, index=False)
            print('Saved:', path.name, len(result), 'rows')
        except Exception:
            print('FAILED: forward', subject)
            traceback.print_exc()
            raise

print('Requested V12B stages completed or checkpointed.')


## 11. Aggregation and manuscript-ready summaries


In [ ]:
checkpoint_files = sorted(
    path for path in (OUTPUT_ROOT / 'checkpoints').glob('*.csv')
    if V12B_CHECKPOINT_RE.fullmatch(path.name) is not None)
if not checkpoint_files:
    raise FileNotFoundError('No V12B checkpoint CSV files found')
folds = pd.concat([pd.read_csv(p) for p in checkpoint_files], ignore_index=True)
if folds['Stage'].isna().any():
    raise AssertionError('A filtered V12B checkpoint has missing Stage')

# Reuse seed-42 focal rows from all21 as the first multi-seed replicate.
if ('all21' in set(folds['Stage'])
        and not ((folds['Stage'].eq('multiseed'))
                 & (folds['Seed'].eq(RANDOM_SEED))).any()):
    focal_keys = pd.DataFrame(FOCAL_PAIRS, columns=['Model', 'FeatureSet'])
    seed42 = folds.loc[folds['Stage'].eq('all21')].merge(
        focal_keys, on=['Model', 'FeatureSet'], how='inner')
    seed42['Stage'] = 'multiseed'
    folds = pd.concat([folds, seed42], ignore_index=True)

folds.to_csv(OUTPUT_ROOT / 'V12B_all_fold_results.csv', index=False)


def pooled_mcc(group):
    # Closed-form binary MCC from pooled confusion counts.
    tn, fp, fn, tp = [float(group[c].sum()) for c in ['TN','FP','FN','TP']]
    den = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den else np.nan


summary_rows = []
group_cols = ['Stage', 'Seed', 'Model', 'FeatureSet']
for keys, group in folds.groupby(group_cols, sort=True):
    stage, seed, model, feature = keys
    seizures = len(group)
    false = int(group['FalseAlarms'].sum())
    hours = float(group['EligibleInterictalHours'].sum())
    summary_rows.append({
        'Stage': stage, 'Seed': int(seed), 'Model': model, 'FeatureSet': feature,
        'Subjects': group['Subject'].nunique(), 'Seizures': seizures,
        'Predicted_SPH0': int(group['SeizurePredicted_SPH0'].sum()),
        'Sensitivity_SPH0': group['SeizurePredicted_SPH0'].mean(),
        'Predicted_MWT5': int(group['SeizurePredicted_MWT5'].sum()),
        'Sensitivity_MWT5': group['SeizurePredicted_MWT5'].mean(),
        'FalseAlarms': false, 'EligibleInterictalHours': hours,
        'FAR_per_h': false/hours if hours else np.nan,
        'TotalAlarms': int(group['TotalAlarms'].sum()),
        'TotalAlarmRate_per_h': group['TotalAlarms'].sum()/group['FoldStreamHours'].sum(),
        'MeanTTW_SPH0_min': group['TTW_SPH0_min'].mean(),
        'MedianTTW_SPH0_min': group['TTW_SPH0_min'].median(),
        'PooledMCC': pooled_mcc(group),
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_ROOT / 'V12B_stage_seed_summary.csv', index=False)

all21 = summary.loc[summary['Stage'].eq('all21')].sort_values(
    ['Sensitivity_MWT5', 'FAR_per_h'], ascending=[False, True])
all21.to_csv(OUTPUT_ROOT / 'V12B_all21_online_safe_summary.csv', index=False)

multiseed = summary.loc[summary['Stage'].eq('multiseed')]
if not multiseed.empty:
    multiseed_summary = multiseed.groupby(['Model','FeatureSet'], as_index=False).agg(
        Seeds=('Seed','nunique'),
        Sensitivity_SPH0_mean=('Sensitivity_SPH0','mean'),
        Sensitivity_SPH0_SD=('Sensitivity_SPH0','std'),
        Sensitivity_MWT5_mean=('Sensitivity_MWT5','mean'),
        Sensitivity_MWT5_SD=('Sensitivity_MWT5','std'),
        FAR_per_h_mean=('FAR_per_h','mean'),
        FAR_per_h_SD=('FAR_per_h','std'),
        PooledMCC_mean=('PooledMCC','mean'),
        PooledMCC_SD=('PooledMCC','std'),
    )
    multiseed_summary.to_csv(OUTPUT_ROOT / 'V12B_multiseed_summary.csv', index=False)
    display(multiseed_summary)

robustness = summary.loc[summary['Stage'].isin(['all21','guard10','nogamma','forward'])]
robustness.to_csv(OUTPUT_ROOT / 'V12B_robustness_comparison.csv', index=False)

manifest = {
    'name': 'MBEC V12B robustness evaluation',
    'stages_present': sorted(folds['Stage'].unique().tolist()),
    'subjects': sorted(folds['Subject'].unique().tolist()),
    'all21_pairs_expected': len(ALL21_PAIRS),
    'focal_pairs': FOCAL_PAIRS,
    'guard_minutes': GUARD_MIN,
    'minimum_prior_seizures_forward': MIN_PRIOR_SEIZURES,
    'multi_seeds_expected': [42] + MULTI_SEEDS_EXTRA,
    'formal_SPH5_SOP30_performed': False,
    'formal_SPH5_note': 'Requires relabeling -35 to -5 min and segmentation rebuild',
}
(OUTPUT_ROOT / 'V12B_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8')

display(all21)
display(robustness)
print('V12B COMPLETE:', OUTPUT_ROOT)
